# Generating RAG Answers

Now we evaluate the full RAG pipeline. 

For each generated question, we: 
- run RAG
- save the answer produced by the LLM 
- compare the answer to the original answer

This form of evaluation is known as A->Q->A': 
- A = original *answer* in the knowledge base 
- Q = generated *question* from this answer
- A' = *answer* generated by our RAG system 
 
For us, we'll be looking at the FAQ's and Answers from LLM-Zoomcamp. 


# Loading the data 

## Load the ground truth questions:

In [1]:
import pandas as pd 

df_ground_truth = pd.read_csv('data/ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'document': '489dd1c9d9'}

### FAQ documents and the search index: 

In [3]:
from utils.ingest import load_faq_data, build_index 

documents = load_faq_data()

documents_llm = [doc for doc in documents if doc["course"]=="llm-zoomcamp"] 

index = build_index(documents=documents_llm)

Create a lookup table for the original FAQ documents:

In [4]:
doc_idx = {document["id"]: document for document in documents_llm}

In [5]:
doc_idx["489dd1c9d9"]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

## Running RAG

In [6]:
from dotenv import load_dotenv 
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
from utils.evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course = "llm-zoomcamp",
)

/Users/evan/Projects/llm-zoomcamp-2026-code/module_04_evaluation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Run RAG for a single question and compute the cost. 

In [8]:
rec = ground_truth[10]
rec

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'document': '489dd1c9d9'}

In [9]:
question = rec['question']

answer_llm = assistant.rag(question)

In [10]:
print(f"Question:\n{question}")
print()
print(f"Answer:\n{answer_llm}")

Question:
How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?

Answer:
Students join via **YouTube Live**, not the public Zoom link. The Zoom link is only shared with instructors/presenters/TAs.  

To participate:
- Watch the live stream on the **DataTalksClub YouTube channel**
- Ask questions in **Slido** (the link is pinned in chat when live)
- Check the **announcements channel on Telegram and Slack** for the video URL before the session starts

Do **not** post questions in chat, since they may be missed.


### How much did that cost? 

In [11]:
assistant.total_cost()

0.0010904999999999999

### Get the original answer from the document ID

In [17]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

### Save it in one record

In [18]:
rag_result = {
    'question': question,
    'answer_llm':answer_llm,
    'answer_orig':answer_orig,
    'document':doc_id 
}

rag_result

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'answer_llm': 'Students join via **YouTube Live**, not the public Zoom link. The Zoom link is only shared with instructors/presenters/TAs.  \n\nTo participate:\n- Watch the live stream on the **DataTalksClub YouTube channel**\n- Ask questions in **Slido** (the link is pinned in chat when live)\n- Check the **announcements channel on Telegram and Slack** for the video URL before the session starts\n\nDo **not** post questions in chat, since they may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).

## Processing all question

In [19]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec['document']
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc['answer']

    result = {
        "question":question,
        "answer_llm":answer_llm,
        "answer_orig":answer_orig,
        "document":doc_id,
    }

    return result

In [20]:
# Test on single result

answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

It works!!! 

So, we'll run it over everything but first we're reset the usage and make use of parallelization.   

In [21]:
assistant.reset_usage()

from concurrent.futures import ThreadPoolExecutor 
from utils.evaluation_utils import map_progress 

### Run RAG for all ground truth questions

In [22]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

100%|██████████| 515/515 [02:22<00:00,  3.62it/s]


In [23]:
answers = [answer_record for answer_record in results]

##### Calculate the cost

In [24]:
assistant.total_cost()

0.5346142500000003

#### Save the answers 

In [25]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

# A->Q->A' Evaluation

Now we compare the RAG answer with the original answer from the FAQ to check if the RAG pipeline is producing answers which match the ground truth. 

In [34]:
from pydantic import BaseModel, Field
from typing import Literal

from utils.evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress


In [35]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer." 
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' if otherwise."
    )

In [36]:
# Prompt 
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()


# prompt template we pass to the judge for each answer. 

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [37]:
rec = answers[0]

In [38]:
rec

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}